In [2]:
import pandas as pd
DATA_FOLDER = '/net/tscratch/people/tutorial237/EnsembleAi2026/task3/data'

# Read just first 100k rows of the raw CSV for May 2025
chunk = next(pd.read_csv(f"{DATA_FOLDER}/data.csv", chunksize=100_000).__iter__())
chunk["year"]  = chunk["timedate"].str[:4].astype("int16")
chunk["month"] = chunk["timedate"].str[5:7].astype("int8")

may_2025 = chunk[(chunk["year"]==2025) & (chunk["month"]==5)]
print(f"May 2025 rows in first chunk: {len(may_2025)}")
print(f"x2 null rate: {may_2025['x2'].isna().mean():.3f}")
print(f"x2 sample values: {may_2025['x2'].dropna().head(10).values}")

May 2025 rows in first chunk: 8920
x2 null rate: 1.000
x2 sample values: []


In [4]:
chunk = next(pd.read_csv(f"{DATA_FOLDER}/data.csv", chunksize=500_000).__iter__())

# Only training rows where x2 is known
train_chunk = chunk[chunk["x2"].notna() & (chunk["x2"] > 0)]

print("5-minute level correlations with x2:")
cols = ["t1","t2","t4","t5","t6","t7","t8","t9","t10","t11","t12","t13","x1"]
corr = train_chunk[cols + ["x2"]].corr()["x2"].drop("x2").sort_values(ascending=False)
print(corr.round(3).to_string())

5-minute level correlations with x2:
t4     0.722
t5     0.681
t13    0.628
t8     0.560
t6     0.479
t12    0.459
t9     0.449
t7     0.007
t11   -0.237
t2    -0.255
t10   -0.275
t1    -0.358
x1       NaN


In [5]:
chunk = next(pd.read_csv(f"{DATA_FOLDER}/data.csv", chunksize=500_000).__iter__())
chunk["year"]  = chunk["timedate"].str[:4].astype("int16")
chunk["month"] = chunk["timedate"].str[5:7].astype("int8")

train_chunk = chunk[chunk["year"].isin([2024]) | 
                    ((chunk["year"]==2025) & (chunk["month"]<=4))]

# For each device-month: what fraction of time is x2 > 0?
running = train_chunk.groupby(["deviceId","year","month"]).agg(
    x2_mean       = ("x2", "mean"),
    x2_running    = ("x2", lambda s: (s > 0).mean()),   # fraction device is ON
    x2_when_on    = ("x2", lambda s: s[s > 0].mean() if (s > 0).any() else 0),  # mean when running
).reset_index()

print("Correlation of components with x2_mean:")
print(running[["x2_mean","x2_running","x2_when_on"]].corr()["x2_mean"].round(3))

print("\nx2_mean decomposition:")
print("x2_mean ≈ x2_running × x2_when_on")
running["reconstructed"] = running["x2_running"] * running["x2_when_on"]
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(running["x2_mean"], running["reconstructed"])
print(f"Reconstruction MAE: {mae:.6f}")

Correlation of components with x2_mean:
x2_mean       1.000
x2_running    0.566
x2_when_on    0.728
Name: x2_mean, dtype: float64

x2_mean decomposition:
x2_mean ≈ x2_running × x2_when_on
Reconstruction MAE: 0.000000


In [8]:
chunk = next(pd.read_csv(f"{DATA_FOLDER}/data.csv", chunksize=500_000).__iter__())
chunk["year"]  = chunk["timedate"].str[:4].astype("int16")
chunk["month"] = chunk["timedate"].str[5:7].astype("int8")

may_2025 = chunk[(chunk["year"]==2025) & (chunk["month"]==5)]
print(f"May 2025 rows: {len(may_2025)}")
print(f"t4 null rate : {may_2025['t4'].isna().mean():.3f}")
print(f"t5 null rate : {may_2025['t5'].isna().mean():.3f}")
print(f"t1 null rate : {may_2025['t1'].isna().mean():.3f}")
print(f"x2 null rate : {may_2025['x2'].isna().mean():.3f}")
print(f"\nSample t4 values: {may_2025['t4'].dropna().head(5).values}")

May 2025 rows: 39123
t4 null rate : 0.000
t5 null rate : 0.000
t1 null rate : 0.000
x2 null rate : 1.000

Sample t4 values: [0.35 0.35 0.35 0.34 0.34]


In [ ]:
v_optuna = pd.read_csv("data/submission_optuna.csv").rename(columns={"prediction":"p_optuna"})
v_5min   = pd.read_csv("data/submission_perdevice_5min.csv").rename(columns={"prediction":"p_5min"})

blend = v_optuna.merge(v_5min, on=["deviceId","year","month"])

for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    blend[f"blend_{w}"] = (w * blend["p_optuna"] + (1-w) * blend["p_5min"]).clip(0,1)
    print(f"  w={w} optuna: mean={blend[f'blend_{w}'].mean():.4f}")

# Submit the 70/30 blend (favour optuna since it scored better)
blend["prediction"] = (0.7 * blend["p_optuna"] + 0.3 * blend["p_5min"]).clip(0,1)
blend[["deviceId","year","month","prediction"]].to_csv("data/submission_blend.csv", index=False)
print("\nSaved → submission_blend_final.csv")